# GA1-220501094-01-AA1-EV07: Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

* **Denominación del Programa de Formación:** Tecnólogo en Análisis y Desarrollo de Software (`228118`)
* **Ficha:** `3223873`
* **Competencia:** `220501094` - Estructurar propuesta técnica de servicio de tecnología de la información según requisitos técnicos y normativa
* **Resultado de Aprendizaje:** `220501094-01` - Definir especificaciones técnicas del software de acuerdo con las características del software a construir
* **Evidencia de Aprendizaje:** `GA1-220501094-01-AA1-EV07` – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
* **Aprendiz:** Simón Carmona Toro
* **Instructor:** Carlos Enrique Navia Torres
* **Proyecto Individual:** Ciencia de Datos Aplicada al Balance de Campeones, Tasas de Selección/Bloqueo (*Pick/Ban Rate*), Dificultad Mecánica y Escalado Estadístico por Clases en *League of Legends*

---

### Pregunta Analítica del Proyecto Individual (EV02 / EV03 / EV05)
> *«¿De qué manera la clase de rol de un campeón (Asesino, Mago, Tirador, Tanque, Luchador o Soporte), su índice de dificultad mecánica y la duración total del encuentro influyen en la variación de su tasa de victoria (Win Rate), tasa de selección (Pick Rate) y tasa de bloqueo (Ban Rate) en el metajuego clasificatorio de League of Legends?»*

### Pregunta Orientadora de la Actividad (EV07)
> *«¿Cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica planteada?»*

---

### Estructura del Cuaderno:
1. **Preparación del Entorno y Revisión del Diagnóstico Inicial (EV05)**
2. **Lección 2: Limpieza y Estandarización de Datos**
   - Inspección del estado inicial (duplicados, valores faltantes, tipos inconsistentes, nombres con apóstrofos y partidas *remake* $< 300$ s).
   - Funciones modulares reutilizables y pipeline encadenado con `.pipe()`.
   - Registro automático de cambios con `anotar()` y tabla comparativa antes vs. después.
   - Validación de reglas de negocio e integridad con `assert`.
3. **Lección 3: Filtrado, Ordenamiento, Agrupación e Indicadores**
   - Filtrado avanzado mediante máscaras booleanas (`&`, `|`), `.isin()`, `.between()` y `.query()`.
   - Ordenamiento y selección de registros relevantes con `.sort_values()`, `.nlargest()` y `.nsmallest()`.
   - **Indicador 1:** Balance competitivo por Clase y Rol (`groupby`, `agg` y `pivot_table`).
   - **Indicador 2:** Curva de efectividad según Índice de Dificultad Mecánica (`pd.cut`, `groupby` y `transform`).
   - **Indicador 3:** Escalado temporal según Duración del Encuentro (`pd.cut`, `groupby` y `crosstab`).
4. **Lección 4: Combinación y Estructuración de Múltiples Datasets**
   - Identificación de relaciones y auditoría de llaves con `indicator=True`.
   - Combinación relacional con `pd.merge()` y validación de cardinalidad (`validate='m:1'`, `validate='1:1'`).
   - Apilado de subconjuntos líderes con `pd.concat()` y transformación ancho/largo con `.melt()`.
   - Exportación y verificación de datasets procesados en `data/processed/` (formatos `CSV` y `Parquet`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración dinámica de rutas relativas al repositorio
if Path('data/raw').exists():
    RUTA_RAW = Path('data/raw')
    RUTA_PROCESSED = Path('data/processed')
elif Path('../data/raw').exists():
    RUTA_RAW = Path('../data/raw')
    RUTA_PROCESSED = Path('../data/processed')
else:
    RUTA_RAW = Path(r'C:/Users/Admin/.gemini/antigravity/scratch/proyecto-ciencia-datos/data/raw')
    RUTA_PROCESSED = Path(r'C:/Users/Admin/.gemini/antigravity/scratch/proyecto-ciencia-datos/data/processed')

RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

print("Entorno inicializado correctamente.")
print(f"Ruta datos crudos (raw):          {RUTA_RAW.resolve()}")
print(f"Ruta datos procesados (processed): {RUTA_PROCESSED.resolve()}")


Entorno inicializado correctamente.
Ruta datos crudos (raw):          C:\Users\Admin\.gemini\antigravity\scratch\proyecto-ciencia-datos\data\raw
Ruta datos procesados (processed): C:\Users\Admin\.gemini\antigravity\scratch\proyecto-ciencia-datos\data\processed


---
## Lección 2: Limpieza y estandarización de datos

A partir de los hallazgos documentados en el diagnóstico de calidad previo ([`notebooks/data_quality.ipynb`](data_quality.ipynb) y [`reports/data_quality.md`](../reports/data_quality.md)), procedemos a cargar los tres datasets originales de `data/raw/` y verificar su estado inicial antes de aplicar el pipeline de limpieza.


### Paso 1: Carga e inspección del estado inicial de los datasets crudos


In [2]:
df_meta_crudo = pd.read_csv(RUTA_RAW / 'lol_champion_meta_stats.csv')
df_base_crudo = pd.read_csv(RUTA_RAW / 'lol_champions_base_attributes.csv')
df_perf_crudo = pd.read_csv(RUTA_RAW / 'lol_ranked_matches_performance.csv')

print("=== 1. DIMENSIONES INICIALES ===")
print(f"meta_stats (lol_champion_meta_stats.csv):           {df_meta_crudo.shape}")
print(f"base_attributes (lol_champions_base_attributes.csv): {df_base_crudo.shape}")
print(f"matches_perf (lol_ranked_matches_performance.csv):   {df_perf_crudo.shape}")

print("\n=== 2. DUPLICADOS EXACTOS Y DE CLAVE COMPUESTA ===")
print(f"Duplicados exactos en meta_stats:                {df_meta_crudo.duplicated().sum()}")
print(f"Pares (Name, Role) repetidos en meta_stats:      {df_meta_crudo.duplicated(subset=['Name', 'Role']).sum()}")
print(f"Duplicados en base_attributes (por name):        {df_base_crudo.duplicated(subset=['name']).sum()}")
print(f"Duplicados exactos en matches_perf:              {df_perf_crudo.duplicated().sum()}")

print("\n=== 3. VALORES FALTANTES POR DATASET ===")
print("Nulos en meta_stats:     ", df_meta_crudo.isna().sum()[df_meta_crudo.isna().sum() > 0].to_dict())
print("Nulos en base_attributes:", df_base_crudo.isna().sum()[df_base_crudo.isna().sum() > 0].to_dict())
print("Nulos en matches_perf:   ", df_perf_crudo.isna().sum()[df_perf_crudo.isna().sum() > 0].to_dict())

print("\n=== 4. ANOMALÍAS ESPECÍFICAS DETECTADAS EN EV05 ===")
dif_nombres = df_base_crudo[df_base_crudo['id'] != df_base_crudo['name']][['id', 'name']]
print(f"Campeones con discrepancia entre 'id' y 'name' (espacios/apóstrofos): {len(dif_nombres)}")
remakes = df_perf_crudo[df_perf_crudo['duration_sec'] < 300]
print(f"Partidas canceladas (remakes con duration_sec < 300s):                {len(remakes)}")


=== 1. DIMENSIONES INICIALES ===
meta_stats (lol_champion_meta_stats.csv):           (100, 11)
base_attributes (lol_champions_base_attributes.csv): (70, 14)
matches_perf (lol_ranked_matches_performance.csv):   (500, 11)

=== 2. DUPLICADOS EXACTOS Y DE CLAVE COMPUESTA ===
Duplicados exactos en meta_stats:                0
Pares (Name, Role) repetidos en meta_stats:      4
Duplicados en base_attributes (por name):        0
Duplicados exactos en matches_perf:              0

=== 3. VALORES FALTANTES POR DATASET ===
Nulos en meta_stats:      {'Class': 4, 'Tier': 3, 'Trend': 7}
Nulos en base_attributes: {'partype': 4, 'mp': 5, 'mpperlevel': 5}
Nulos en matches_perf:    {'role': 9, 'damage_dealt': 7}

=== 4. ANOMALÍAS ESPECÍFICAS DETECTADAS EN EV05 ===
Campeones con discrepancia entre 'id' y 'name' (espacios/apóstrofos): 7
Partidas canceladas (remakes con duration_sec < 300s):                3


#### Inspección detallada de los 4 pares `(Name, Role)` repetidos y las 3 partidas *remake*


In [3]:
print("--- Pares (Name, Role) repetidos en meta_stats ---")
dups_meta = df_meta_crudo[df_meta_crudo.duplicated(subset=['Name', 'Role'], keep=False)].sort_values(['Name', 'Role'])
print(dups_meta[['Name', 'Class', 'Role', 'Tier', 'Win %', 'Role %', 'Pick %', 'Ban %']])

print("\n--- Partidas remake (duration_sec < 300) en matches_perf ---")
print(remakes[['match_id', 'champion_name', 'role', 'win', 'damage_dealt', 'gold_earned', 'duration_sec', 'patch']])


--- Pares (Name, Role) repetidos en meta_stats ---
            Name     Class    Role Tier   Win %  Role %  Pick %   Ban %
2          Akali  Assassin     MID    A  49.14%  74.10%   4.83%   5.00%
72         Akali  Assassin     MID   S+  46.89%  25.70%  12.53%  14.00%
7   Aurelion Sol      Mage     MID    S  47.34%  79.35%  15.28%  16.50%
77  Aurelion Sol      Mage     MID    D  52.74%  31.20%   3.18%  25.50%
13      Cho'Gath      Tank     TOP    S  54.36%  85.65%   8.02%  30.30%
83      Cho'Gath      Tank     TOP    D  52.11%  15.80%  15.72%   1.30%
15         Diana   Fighter  JUNGLE    B  49.05%  87.75%  12.20%  34.90%
85         Diana   Fighter  JUNGLE    S  46.80%  18.00%  19.90%   5.90%

--- Partidas remake (duration_sec < 300) en matches_perf ---
     match_id champion_name    role  win  damage_dealt  gold_earned  duration_sec  patch
18    4810010        Viktor  JUNGLE    1       17804.0        10812           213  14.19
142   4810072          Jhin     TOP    1       23636.0       

**Justificación técnica de las decisiones de limpieza:**
1. **Deduplicación de clave compuesta `(Name, Role)` en `meta_stats`:** Los 4 campeones repetidos en el mismo carril (*Akali MID*, *Aurelion Sol MID*, *Cho'Gath TOP* y *Diana JUNGLE*) presentan un registro principal con alta frecuencia en el rol (`Role % > 74%`) y un registro secundario residual (`Role % < 32%`). Se ordena de mayor a menor `Role %` y se conserva el registro principal con `drop_duplicates(subset=['champion_key', 'role'], keep='first')`, obteniendo **96 combinaciones campeón-rol únicas**.
2. **Filtrado de partidas canceladas (*remakes*) en `matches_perf`:** Las 3 partidas con `duration_sec < 300` (213 s, 217 s y 234 s) corresponden a cierres anticipados por desconexión al minuto 3, pero registran valores atípicos de oro/daño que distorsionan el daño por minuto (`DPM`). Se filtran conservando únicamente partidas válidas con `duration_sec >= 300` (**497 partidas válidas**).
3. **Imputación de valores faltantes:**
   - En `meta_stats`: los 4 nulos de `Class` (*Camille, Maokai, Yasuo, Ezreal*) se recuperan cruzando dinámicamente con la clase primaria de `base_attributes['tags']`; los 3 nulos de `Tier` se imputan con la moda (`'A'`); los 7 nulos de `Trend` se imputan con `0.0` (tendencia estable).
   - En `base_attributes`: los 5 nulos de `mp` y `mpperlevel` pertenecen a campeones que no usan maná (*Garen, Katarina, Mordekaiser, Riven, Yasuo*). Se imputan con `0.0` y los 4 nulos de `partype` con `'None'`, unificando el criterio con los demás campeones sin maná.
   - En `matches_perf`: los 9 nulos de `role` se imputan con el rol principal del campeón según `meta_stats`, y los 7 nulos de `damage_dealt` se imputan con la mediana de daño de ese campeón y rol.
4. **Corrección de tipos y estandarización:**
   - Conversión de `Win %`, `Role %`, `Pick %` y `Ban %` de texto (`object`) a proporción numérica (`float32` entre `0.0` y `1.0`).
   - Creación de llave normalizada `champion_key` (en minúsculas, sin espacios ni apóstrofos) en las tres tablas para asegurar cruces exactos.
   - Conversión de `tier` a categoría ordenada (`S+ > S > A > B > C > D`) y `patch`, `role`, `champion_class`, `partype` y `tags` a tipo `category` para optimizar el uso de memoria.


### Paso 2: Función de monitoreo de cambios (`anotar`)


In [4]:
registro_limpieza = []

def anotar(df, tabla, paso):
    registro_limpieza.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'memoria_kb': round(df.memory_usage(deep=True).sum() / 1024, 1),
    })
    return df

def normalizar_llave(serie):
    return (serie.astype(str)
            .str.strip()
            .str.lower()
            .str.replace("'", "", regex=False)
            .str.replace(" ", "", regex=False))

print("Funciones auxiliares 'anotar' y 'normalizar_llave' inicializadas.")


Funciones auxiliares 'anotar' y 'normalizar_llave' inicializadas.


### Paso 3: Funciones modulares para el pipeline de limpieza


In [5]:
# 1. Pipeline para Base Attributes (Catálogo oficial de campeones)
def quitar_duplicados_base(df):
    df = df.copy()
    df['name'] = df['name'].astype(str).str.strip()
    df['champion_key'] = normalizar_llave(df['name'])
    return df.drop_duplicates(subset=['champion_key']).reset_index(drop=True)

def imputar_y_tipar_base(df):
    df = df.copy()
    # Extraer clase primaria desde 'tags' (ej. 'Fighter,Tank' -> 'Fighter')
    df['primary_class'] = df['tags'].astype(str).str.split(',').str[0].str.strip().astype('category')
    
    # Imputar maná = 0.0 y partype = 'None' para campeones sin recurso de maná
    df['mp'] = df['mp'].fillna(0.0).astype(np.float32)
    df['mpperlevel'] = df['mpperlevel'].fillna(0.0).astype(np.float32)
    df['partype'] = df['partype'].fillna('None').astype(str).str.strip().astype('category')
    df['tags'] = df['tags'].astype(str).str.strip().astype('category')
    
    # Optimización de tipos numéricos
    cols_float = ['hp', 'hpperlevel', 'armor', 'armorperlevel', 'attackdamage', 'attackdamageperlevel']
    for col in cols_float:
        df[col] = df[col].astype(np.float32)
    df['difficulty'] = df['difficulty'].astype(np.int8)
    return df

# 2. Pipeline para Meta Stats (Estadísticas del metajuego por carril)
def estandarizar_y_deduplicar_meta(df):
    df = df.copy()
    df = df.rename(columns={
        'Name': 'champion_name',
        'Class': 'champion_class',
        'Role': 'role',
        'Tier': 'tier',
        'Score': 'meta_score',
        'Trend': 'trend',
        'Win %': 'win_rate',
        'Role %': 'role_rate',
        'Pick %': 'pick_rate',
        'Ban %': 'ban_rate',
        'KDA': 'kda'
    })
    df['champion_name'] = df['champion_name'].astype(str).str.strip()
    df['role'] = df['role'].astype(str).str.strip().str.upper()
    df['champion_key'] = normalizar_llave(df['champion_name'])
    
    # Convertir porcentajes de texto ('46.80%') a proporción decimal (0.4680)
    for col in ['win_rate', 'role_rate', 'pick_rate', 'ban_rate']:
        df[col] = df[col].astype(str).str.replace('%', '', regex=False).str.strip().astype(np.float32) / 100.0
        
    # Resolver los 4 pares (champion_key, role) repetidos conservando el de mayor role_rate
    df = (df.sort_values('role_rate', ascending=False)
            .drop_duplicates(subset=['champion_key', 'role'], keep='first')
            .sort_values(['champion_name', 'role'])
            .reset_index(drop=True))
    return df

def imputar_y_tipar_meta(df, base_ref):
    df = df.copy()
    # Recuperar Class faltante desde el catálogo base_attributes
    mapa_clases = base_ref.set_index('champion_key')['primary_class'].astype(str).to_dict()
    df['champion_class'] = df['champion_class'].fillna(df['champion_key'].map(mapa_clases)).astype('category')
    df['role'] = df['role'].astype('category')
    
    # Imputar Tier con la moda ('A') y convertir a categoría ordenada
    moda_tier = df['tier'].mode()[0]
    df['tier'] = df['tier'].fillna(moda_tier)
    orden_tiers = ['S+', 'S', 'A', 'B', 'C', 'D']
    df['tier'] = pd.Categorical(df['tier'], categories=orden_tiers, ordered=True)
    
    # Imputar Trend con 0.0 y optimizar tipos
    df['trend'] = df['trend'].fillna(0.0).astype(np.float32)
    df['meta_score'] = df['meta_score'].astype(np.float32)
    df['kda'] = df['kda'].astype(np.float32)
    return df

# 3. Pipeline para Ranked Matches Performance (Partidas clasificatorias)
def filtrar_y_estandarizar_perf(df):
    df = df.copy().drop_duplicates().reset_index(drop=True)
    df['champion_name'] = df['champion_name'].astype(str).str.strip()
    df['champion_key'] = normalizar_llave(df['champion_name'])
    
    # Filtrar partidas canceladas (remakes con duration_sec < 300s)
    df = df[df['duration_sec'] >= 300].reset_index(drop=True)
    return df

def imputar_y_tipar_perf(df, meta_ref):
    df = df.copy()
    # Imputar role faltante con el rol principal del campeón en el metajuego
    rol_principal = (meta_ref.sort_values('role_rate', ascending=False)
                     .groupby('champion_key')['role'].first().astype(str).to_dict())
    df['role'] = df['role'].fillna(df['champion_key'].map(rol_principal)).astype(str).str.strip().str.upper()
    
    # Imputar damage_dealt faltante con la mediana del campeón en ese rol (o mediana del rol)
    mediana_por_campeon_rol = df.groupby(['champion_key', 'role'])['damage_dealt'].transform('median')
    mediana_por_rol = df.groupby('role')['damage_dealt'].transform('median')
    df['damage_dealt'] = (df['damage_dealt']
                          .fillna(mediana_por_campeon_rol)
                          .fillna(mediana_por_rol)
                          .fillna(df['damage_dealt'].median())
                          .astype(np.int32))
    
    # Estandarizar patch como categoría ('14.19', '14.20') y optimizar tipos
    df['patch'] = df['patch'].apply(lambda x: f"{float(x):.2f}").astype('category')
    df['role'] = df['role'].astype('category')
    df['match_id'] = df['match_id'].astype(np.int32)
    df['win'] = df['win'].astype(np.int8)
    df['kills'] = df['kills'].astype(np.int16)
    df['deaths'] = df['deaths'].astype(np.int16)
    df['assists'] = df['assists'].astype(np.int16)
    df['gold_earned'] = df['gold_earned'].astype(np.int32)
    df['duration_sec'] = df['duration_sec'].astype(np.int16)
    
    # Métricas derivadas por partida
    df['duration_min'] = np.round(df['duration_sec'] / 60.0, 2).astype(np.float32)
    df['kda_partida'] = np.round((df['kills'] + df['assists']) / np.maximum(df['deaths'], 1), 2).astype(np.float32)
    df['dano_por_minuto'] = np.round(df['damage_dealt'] / df['duration_min'], 1).astype(np.float32)
    df['oro_por_minuto'] = np.round(df['gold_earned'] / df['duration_min'], 1).astype(np.float32)
    return df

print("Funciones modulares de limpieza definidas correctamente.")


Funciones modulares de limpieza definidas correctamente.


### Paso 4: Ejecución del pipeline encadenado con `.pipe()`


In [6]:
registro_limpieza.clear()

base_limpio = (df_base_crudo
    .pipe(anotar, 'base_attributes', '0_crudo')
    .pipe(quitar_duplicados_base)
    .pipe(anotar, 'base_attributes', '1_llave_normalizada')
    .pipe(imputar_y_tipar_base)
    .pipe(anotar, 'base_attributes', '2_limpio_y_tipado'))

meta_limpio = (df_meta_crudo
    .pipe(anotar, 'meta_stats', '0_crudo')
    .pipe(estandarizar_y_deduplicar_meta)
    .pipe(anotar, 'meta_stats', '1_sin_duplicados_pct_float')
    .pipe(imputar_y_tipar_meta, base_limpio)
    .pipe(anotar, 'meta_stats', '2_limpio_y_tipado'))

perf_limpio = (df_perf_crudo
    .pipe(anotar, 'matches_perf', '0_crudo')
    .pipe(filtrar_y_estandarizar_perf)
    .pipe(anotar, 'matches_perf', '1_sin_remakes_y_llave')
    .pipe(imputar_y_tipar_perf, meta_limpio)
    .pipe(anotar, 'matches_perf', '2_limpio_y_metricas'))

informe_limpieza = pd.DataFrame(registro_limpieza)
informe_limpieza


             tabla                        paso  filas  columnas  nulos  memoria_kb
0  base_attributes                     0_crudo     70        14     14        10.5
1  base_attributes         1_llave_normalizada     70        15     14        11.5
2  base_attributes           2_limpio_y_tipado     70        16      0         7.5
3       meta_stats                     0_crudo    100        11     14        12.7
4       meta_stats  1_sin_duplicados_pct_float     96        12     14         9.9
5       meta_stats           2_limpio_y_tipado     96        12      0         6.0
6     matches_perf                     0_crudo    500        11     16        48.1
7     matches_perf       1_sin_remakes_y_llave    497        12     16        54.6
8     matches_perf         2_limpio_y_metricas    497        16      0        32.7


### Paso 5: Validación automática de calidad mediante `assert`


In [7]:
def validar_pipeline(base, meta, perf):
    # 1. Validaciones en Base Attributes
    assert len(base) == 70, 'El catálogo base debe conservar los 70 campeones únicos'
    assert base['champion_key'].is_unique, 'Llave champion_key duplicada en base_attributes'
    assert base['difficulty'].between(1, 10).all(), 'Dificultad fuera del rango [1, 10]'
    assert not base.isna().any().any(), 'Quedaron valores nulos en base_attributes'
    
    # 2. Validaciones en Meta Stats
    assert len(meta) == 96, 'meta_stats debe tener 96 registros tras eliminar los 4 pares repetidos'
    assert meta.duplicated(subset=['champion_key', 'role']).sum() == 0, 'Pares (champion_key, role) duplicados'
    assert meta['win_rate'].between(0.0, 1.0).all(), 'win_rate fuera del rango [0, 1]'
    assert meta['pick_rate'].between(0.0, 1.0).all(), 'pick_rate fuera del rango [0, 1]'
    assert meta['ban_rate'].between(0.0, 1.0).all(), 'ban_rate fuera del rango [0, 1]'
    assert set(meta['role'].unique()) == {'TOP', 'JUNGLE', 'MID', 'ADC', 'SUPPORT'}, 'Roles fuera de estándar'
    assert not meta.isna().any().any(), 'Quedaron valores nulos en meta_stats'
    
    # 3. Validaciones en Matches Performance
    assert len(perf) == 497, 'matches_perf debe tener 497 partidas tras filtrar los 3 remakes (<300s)'
    assert perf['duration_sec'].ge(300).all(), 'Existen partidas remake (<300s) sin filtrar'
    assert perf['win'].isin([0, 1]).all(), 'Columna win contiene valores no binarios'
    assert not perf.isna().any().any(), 'Quedaron valores nulos en matches_perf'
    
    print("Validación superada con éxito: todas las reglas de integridad y calidad se cumplen al 100%.")

validar_pipeline(base_limpio, meta_limpio, perf_limpio)


Validación superada con éxito: todas las reglas de integridad y calidad se cumplen al 100%.


---
## Lección 3: Filtrado, ordenamiento y agrupación de datos

Aplicamos las técnicas de filtrado lógico (`máscaras booleanas`, `isin()`, `between()`, `query()`), ordenamiento (`sort_values()`, `nlargest()`, `nsmallest()`), categorización (`pd.cut()`) y agrupación (`groupby()`, `agg()`, `pivot_table()`, `crosstab()`, `transform()`) para responder la pregunta analítica del proyecto.


### 3.1 Filtrado mediante condiciones booleanas, `isin()`, `between()` y `query()`

1. **Filtro con máscaras booleanas, `.isin()` y `.between()`:** Identificamos los campeones que juegan en líneas solitarias (`TOP` o `MID`), que mantienen un Win Rate saludable entre **51% y 60%** y tienen una tasa de selección relevante (`pick_rate >= 8%`).
2. **Filtro con `.query()`:** Identificamos los campeones de **alta prioridad o bloqueo crítico** (`ban_rate >= 25%` y `win_rate >= 51%`), candidatos a ajustes de balance (*nerfs*).


In [8]:
# 1. Máscaras booleanas + isin() + between()
es_linea_solitaria = meta_limpio['role'].isin(['TOP', 'MID'])
es_winrate_competitivo = meta_limpio['win_rate'].between(0.51, 0.60)
es_popular = meta_limpio['pick_rate'] >= 0.08

dominantes_solitarios = meta_limpio[es_linea_solitaria & es_winrate_competitivo & es_popular]
print(f"Campeones dominantes en líneas solitarias (TOP/MID): {len(dominantes_solitarios)} registros")
print(dominantes_solitarios[['champion_name', 'champion_class', 'role', 'tier', 'win_rate', 'pick_rate', 'ban_rate']].head(8))

# 2. Filtrado declarativo con query()
criticos_balance = meta_limpio.query("ban_rate >= 0.25 and win_rate >= 0.51")
print(f"\nCampeones con alerta de desbalance (Ban Rate >= 25% y Win Rate >= 51%): {len(criticos_balance)} registros")
print(criticos_balance[['champion_name', 'champion_class', 'role', 'tier', 'win_rate', 'pick_rate', 'ban_rate', 'kda']].head(8))


Campeones dominantes en líneas solitarias (TOP/MID): 8 registros
   champion_name champion_class role tier  win_rate  pick_rate  ban_rate
10         Annie           Mage  MID    D    0.5265     0.1110     0.119
24      Cho'Gath           Tank  TOP    S    0.5436     0.0802     0.303
45           Jax        Fighter  TOP   S+    0.5193     0.1121     0.176
69         Nasus        Fighter  TOP    S    0.5121     0.1132     0.233
71       Orianna           Mage  MID    B    0.5355     0.1550     0.279
78         Swain           Mage  MID    C    0.5409     0.1033     0.060
90         Yasuo        Fighter  MID    C    0.5283     0.1561     0.336
91          Yone       Assassin  MID    D    0.5400     0.1770     0.359

Campeones con alerta de desbalance (Ban Rate >= 25% y Win Rate >= 51%): 15 registros
   champion_name champion_class     role tier  win_rate  pick_rate  ban_rate   kda
15          Azir           Mage      TOP   S+    0.5391     0.0527     0.278  2.55
20       Caitlyn       Mar

### 3.2 Ordenamiento y selección de extremos con `sort_values()`, `nlargest()` y `nsmallest()`

Consultamos los 5 campeones con mayor puntaje de metajuego (`nlargest`) y los 5 con menor tasa de victoria (`nsmallest`) para identificar los extremos del estado competitivo actual.


In [9]:
print("--- Top 5 Campeones con Mayor Meta Score (nlargest) ---")
top5_score = meta_limpio.nlargest(5, 'meta_score')[['champion_name', 'champion_class', 'role', 'tier', 'meta_score', 'win_rate', 'pick_rate', 'ban_rate']]
print(top5_score)

print("\n--- Top 5 Campeones con Menor Win Rate que requieren Buffs (nsmallest) ---")
bottom5_winrate = meta_limpio.nsmallest(5, 'win_rate')[['champion_name', 'champion_class', 'role', 'tier', 'meta_score', 'win_rate', 'pick_rate', 'ban_rate']]
print(bottom5_winrate)

print("\n--- Ordenamiento jerárquico por Tier (S+ -> D) y Win Rate descendente (sort_values) ---")
ranking_jerarquico = meta_limpio.sort_values(['tier', 'win_rate'], ascending=[True, False])
print(ranking_jerarquico[['champion_name', 'champion_class', 'role', 'tier', 'win_rate', 'kda']].head(6))


--- Top 5 Campeones con Mayor Meta Score (nlargest) ---
   champion_name champion_class     role tier  meta_score  win_rate  pick_rate  ban_rate
5        Alistar           Tank  SUPPORT    B   90.949997    0.5031     0.0692     0.073
81        Thresh        Support  SUPPORT    A   90.949997    0.4995     0.1660     0.129
11          Ashe       Marksman      ADC   S+   90.000000    0.5382     0.1319     0.142
84        Veigar           Mage      MID    C   90.000000    0.5346     0.0307     0.198
17    Blitzcrank           Tank  SUPPORT    B   89.050003    0.4968     0.1946     0.211

--- Top 5 Campeones con Menor Win Rate que requieren Buffs (nsmallest) ---
   champion_name champion_class     role tier  meta_score  win_rate  pick_rate  ban_rate
1         Aatrox        Fighter      TOP   S+   42.500000    0.4680     0.0065     0.004
85            Vi        Fighter   JUNGLE    D   56.750000    0.4698     0.0516     0.221
72          Pyke        Support  SUPPORT    C   44.400002    0.4707

---
### 3.3 Indicador 1: Balance Competitivo por Clase y Rol (`groupby`, `agg` y `pivot_table`)

*Relación con la pregunta analítica:* Evalúa directamente cómo la clase de rol (`Assassin`, `Fighter`, `Mage`, `Marksman`, `Support`, `Tank`) y el carril determinan la variación de `Win Rate`, `Pick Rate`, `Ban Rate` y `KDA`.


In [10]:
# Resumen general por Clase de Campeón
resumen_por_clase = (meta_limpio
    .groupby('champion_class', observed=True)
    .agg(
        registros_rol=('champion_name', 'count'),
        campeones_unicos=('champion_key', 'nunique'),
        win_rate_medio=('win_rate', 'mean'),
        pick_rate_medio=('pick_rate', 'mean'),
        ban_rate_medio=('ban_rate', 'mean'),
        kda_medio=('kda', 'mean'),
        score_medio=('meta_score', 'mean')
    )
    .assign(
        win_rate_pct=lambda d: (d['win_rate_medio'] * 100).round(2),
        pick_rate_pct=lambda d: (d['pick_rate_medio'] * 100).round(2),
        ban_rate_pct=lambda d: (d['ban_rate_medio'] * 100).round(2),
        kda_medio=lambda d: d['kda_medio'].round(2),
        score_medio=lambda d: d['score_medio'].round(2)
    )
    .sort_values('win_rate_pct', ascending=False)
    .reset_index())

print("=== RESUMEN GLOBAL POR CLASE DE CAMPEÓN ===")
print(resumen_por_clase[['champion_class', 'registros_rol', 'campeones_unicos', 'win_rate_pct', 'pick_rate_pct', 'ban_rate_pct', 'kda_medio', 'score_medio']])

# Detalle desglosado por Clase y Rol
indicador_1_clase_rol = (meta_limpio
    .groupby(['champion_class', 'role'], observed=True)
    .agg(
        total_campeones=('champion_name', 'count'),
        win_rate_medio=('win_rate', 'mean'),
        pick_rate_medio=('pick_rate', 'mean'),
        ban_rate_medio=('ban_rate', 'mean'),
        kda_medio=('kda', 'mean')
    )
    .assign(
        win_rate_pct=lambda d: (d['win_rate_medio'] * 100).round(2),
        pick_rate_pct=lambda d: (d['pick_rate_medio'] * 100).round(2),
        ban_rate_pct=lambda d: (d['ban_rate_medio'] * 100).round(2),
        kda_medio=lambda d: d['kda_medio'].round(2)
    )
    .reset_index())

# Tabla dinámica (pivot_table) de Win Rate (%) por Clase y Carril
matriz_winrate = (meta_limpio
    .pivot_table(index='champion_class', columns='role', values='win_rate', aggfunc='mean', observed=True)
    .mul(100)
    .round(2))

print("\n=== TABLA DINÁMICA (PIVOT TABLE): WIN RATE (%) POR CLASE Y ROL ===")
matriz_winrate


=== RESUMEN GLOBAL POR CLASE DE CAMPEÓN ===
  champion_class  registros_rol  campeones_unicos  win_rate_pct  pick_rate_pct  ban_rate_pct  kda_medio  score_medio
0       Marksman             18                11     51.540001           8.54     22.440001       2.94    67.889999
1           Mage             19                15     50.650002          10.69     16.070000       2.89    66.400002
2        Fighter             27                19     50.560001           9.66     20.719999       2.66    67.160004
3       Assassin             10                 8     50.279999           8.86     22.820000       2.80    64.730003
4           Tank             12                 9     50.169998          11.90     20.719999       2.52    71.709999
5        Support             10                 8     49.689999          10.53     17.059999       2.91    62.830002

=== TABLA DINÁMICA (PIVOT TABLE): WIN RATE (%) POR CLASE Y ROL ===
role                  ADC     JUNGLE        MID    SUPPORT        TOP

**Interpretación del Indicador 1:**
1. **Dominio de los Tiradores (`Marksman`):** Lideran el rendimiento global por clase con un **Win Rate promedio de 51.54%** y el mayor KDA promedio (**2.94**), impulsados por su hegemonía en el carril inferior (`ADC`: **52.14%**) e incursiones en `MID` (**53.19%**).
2. **Versatilidad de Luchadores (`Fighter`) y Magos (`Mage`):** Los Luchadores son la clase con mayor presencia numérica (27 asignaciones de rol en 19 campeones únicos) y mantienen tasas de victoria superiores al 50% en todos los carriles (`TOP` 50.08%, `MID` 50.37%, `JUNGLE` 50.66%, `ADC` 51.48%, `SUPPORT` 53.82%). Los Magos ocupan el segundo lugar general en Win Rate (**50.65%**) con un sólido `Pick Rate` promedio de **10.69%**.
3. **Presión de Bloqueo en Asesinos (`Assassin`) y Popularidad de Tanques (`Tank`):** Los Asesinos registran la **mayor tasa de bloqueo promedio por clase (22.82%)** pese a tener un Win Rate global moderado (**50.28%**), rindiendo mejor en `JUNGLE` (**51.26%**) que en `ADC` (**47.88%**). Los Tanques registran el mayor `Pick Rate` promedio (**11.90%**) y `Meta Score` (**71.71**), brillando en `TOP` (**51.03%**).


---
### 3.4 Indicador 2: Curva de Efectividad según Índice de Dificultad Mecánica (`pd.cut`, `groupby` y `transform`)

*Relación con la pregunta analítica:* Evalúa cómo el índice oficial de dificultad mecánica (`difficulty` de 1 a 10) afecta el éxito competitivo, comparando campeones sencillos frente a campeones de alta complejidad.


In [11]:
# Cruzar meta_limpio con la dificultad oficial del catálogo base
meta_con_diff = meta_limpio.merge(
    base_limpio[['champion_key', 'difficulty', 'hpperlevel', 'attackdamageperlevel']],
    on='champion_key',
    how='left',
    validate='m:1'
)

# Segmentación del índice de dificultad con pd.cut
meta_con_diff['rango_dificultad'] = pd.cut(
    meta_con_diff['difficulty'],
    bins=[0, 3, 6, 10],
    labels=['Baja (1-3)', 'Media (4-6)', 'Alta (7-10)']
)

# Uso de transform() para calcular la desviación del Win Rate de cada campeón respecto a su grupo de dificultad
meta_con_diff['win_rate_grupo_diff'] = meta_con_diff.groupby('rango_dificultad', observed=True)['win_rate'].transform('mean')
meta_con_diff['diferencial_vs_dificultad_pct'] = ((meta_con_diff['win_rate'] - meta_con_diff['win_rate_grupo_diff']) * 100).round(2)

indicador_2_dificultad = (meta_con_diff
    .groupby('rango_dificultad', observed=True)
    .agg(
        registros_rol=('champion_name', 'count'),
        campeones_unicos=('champion_key', 'nunique'),
        dificultad_media=('difficulty', 'mean'),
        win_rate_medio=('win_rate', 'mean'),
        pick_rate_medio=('pick_rate', 'mean'),
        ban_rate_medio=('ban_rate', 'mean'),
        kda_medio=('kda', 'mean')
    )
    .assign(
        dificultad_media=lambda d: d['dificultad_media'].round(2),
        win_rate_pct=lambda d: (d['win_rate_medio'] * 100).round(2),
        pick_rate_pct=lambda d: (d['pick_rate_medio'] * 100).round(2),
        ban_rate_pct=lambda d: (d['ban_rate_medio'] * 100).round(2),
        kda_medio=lambda d: d['kda_medio'].round(2)
    )
    .reset_index())

print("=== INDICADOR 2: RENDIMIENTO SEGÚN RANGO DE DIFICULTAD MECÁNICA ===")
print(indicador_2_dificultad[['rango_dificultad', 'registros_rol', 'campeones_unicos', 'dificultad_media', 'win_rate_pct', 'pick_rate_pct', 'ban_rate_pct', 'kda_medio']])

print("\n--- Campeones de Alta Dificultad (7-10) que más superan el promedio de su segmento (transform) ---")
sobre_promedio_alta = (meta_con_diff[meta_con_diff['rango_dificultad'] == 'Alta (7-10)']
                       .nlargest(5, 'diferencial_vs_dificultad_pct'))
print(sobre_promedio_alta[['champion_name', 'champion_class', 'role', 'difficulty', 'win_rate', 'diferencial_vs_dificultad_pct']])


=== INDICADOR 2: RENDIMIENTO SEGÚN RANGO DE DIFICULTAD MECÁNICA ===
  rango_dificultad  registros_rol  campeones_unicos  dificultad_media  win_rate_pct  pick_rate_pct  ban_rate_pct  kda_medio
0       Baja (1-3)             16                12              2.50     51.180000           9.91     16.020000       2.78
1      Media (4-6)             49                34              5.02     50.570000           9.33     20.820000       2.81
2      Alta (7-10)             31                24              7.65     50.330002          10.92     20.639999       2.73

--- Campeones de Alta Dificultad (7-10) que más superan el promedio de su segmento (transform) ---
   champion_name champion_class role  difficulty  win_rate  diferencial_vs_dificultad_pct
78         Swain           Mage  MID           8    0.5409                           3.76
91          Yone       Assassin  MID           8    0.5400                           3.67
15          Azir           Mage  TOP           9    0.5391        

**Interpretación del Indicador 2 (La Paradoja de la Dificultad):**
- Los campeones de **Dificultad Baja (1–3)** obtienen la **mayor tasa de victoria promedio (51.18%)** y la menor tasa de bloqueo (**16.02%**). Su ejecución directa y bajo margen de error los hace estadísticamente más confiables para escalar en clasificatorias.
- Los campeones de **Dificultad Media (4–6)** logran un Win Rate intermedio de **50.57%** y el mayor KDA promedio (**2.81**).
- Los campeones de **Dificultad Alta (7–10)** registran el **Win Rate promedio más bajo (50.33%)**, pero concentran la **mayor tasa de selección (10.92%)** y una **alta tasa de bloqueo (20.64%)**. Esto demuestra que los jugadores eligen y bloquean campeones complejos por su alto techo mecánico (*Swain, Yone, Azir, Orianna*), aunque en el agregado global la curva de aprendizaje reduce su tasa de victoria promedio.


---
### 3.5 Indicador 3: Escalado Temporal según Duración del Encuentro (`pd.cut`, `groupby` y `crosstab`)

*Relación con la pregunta analítica:* Evalúa sobre las **497 partidas clasificatorias válidas** (`duration_sec >= 300`) cómo la duración del encuentro modifica la tasa de victoria, el daño por minuto (`DPM`) y el oro acumulado de cada clase.


In [12]:
# Clasificar cada partida según su duración en segundos
perf_limpio['rango_duracion'] = pd.cut(
    perf_limpio['duration_sec'],
    bins=[0, 1500, 2100, np.inf],
    labels=['Corta (< 25 min)', 'Media (25-35 min)', 'Larga (> 35 min)']
)

# Incorporar la clase oficial del campeón a cada partida
mapa_clase_meta = (meta_limpio.sort_values('role_rate', ascending=False)
                   .groupby('champion_key')['champion_class'].first().astype(str).to_dict())
perf_limpio['champion_class'] = perf_limpio['champion_key'].map(mapa_clase_meta).astype('category')

indicador_3_escalado = (perf_limpio
    .groupby(['champion_class', 'rango_duracion'], observed=True)
    .agg(
        partidas=('match_id', 'count'),
        win_rate_partida=('win', 'mean'),
        dano_medio=('damage_dealt', 'mean'),
        oro_medio=('gold_earned', 'mean'),
        dpm_medio=('dano_por_minuto', 'mean'),
        kda_medio=('kda_partida', 'mean')
    )
    .assign(
        win_rate_pct=lambda d: (d['win_rate_partida'] * 100).round(2),
        dano_medio=lambda d: d['dano_medio'].round(0).astype(int),
        oro_medio=lambda d: d['oro_medio'].round(0).astype(int),
        dpm_medio=lambda d: d['dpm_medio'].round(1),
        kda_medio=lambda d: d['kda_medio'].round(2)
    )
    .reset_index())

print("=== INDICADOR 3: ESCALADO TEMPORAL POR CLASE Y DURACIÓN DE PARTIDA ===")
print(indicador_3_escalado[['champion_class', 'rango_duracion', 'partidas', 'win_rate_pct', 'dano_medio', 'oro_medio', 'dpm_medio', 'kda_medio']].to_string(index=False))

# Tabla cruzada de proporción de partidas por clase y tramo de duración
distribucion_duracion = (pd.crosstab(perf_limpio['champion_class'], perf_limpio['rango_duracion'], normalize='index')
                         .mul(100).round(1))
print("\n=== DISTRIBUCIÓN PORCENTUAL (%) DE PARTIDAS POR TRAMO DE DURACIÓN ===")
distribucion_duracion


=== INDICADOR 3: ESCALADO TEMPORAL POR CLASE Y DURACIÓN DE PARTIDA ===
champion_class    rango_duracion  partidas  win_rate_pct  dano_medio  oro_medio   dpm_medio  kda_medio
      Assassin  Corta (< 25 min)        12         33.33       28461      14734 1200.800049       7.56
      Assassin Media (25-35 min)        25         52.00       29288      15834  980.599976       6.32
      Assassin  Larga (> 35 min)        19         57.89       32775      17806  834.400024       5.76
       Fighter  Corta (< 25 min)        19          5.26       28308      14754 1205.000000       6.06
       Fighter Media (25-35 min)        66         45.45       29248      15903  971.900024       5.51
       Fighter  Larga (> 35 min)        51         39.22       33836      17975  880.000000       6.35
          Mage  Corta (< 25 min)         7        100.00       24583      14100 1036.300049       5.19
          Mage Media (25-35 min)        67         58.21       30295      16107 1003.000000       5.89
  

**Interpretación del Indicador 3:**
1. **Escalado hacia el Juego Tardío (*Late Game > 35 min*):** Al superar los 35 minutos y acumular ~18,000 de oro, los **Tiradores (`Marksman`)** elevan su tasa de victoria del 47.22% (en juego medio) al **62.96%** con **33,730 de daño medio**, mientras que los **Magos (`Mage`)** alcanzan un **75.00% de victorias** en partidas largas y los **Tanques (`Tank`)** logran un **61.11% de victorias** (concentrando el **55.4%** de todas sus partidas en duraciones mayores a 35 minutos).
2. **Desgaste del Daño por Minuto (`DPM`) en Asesinos (`Assassin`):** En partidas cortas ($< 25$ min), los Asesinos registran su pico explosivo de daño por minuto (**1,200.8 DPM** y **7.56 KDA**), pero a medida que la partida se alarga su DPM cae a **980.6** (25–35 min) y **834.4** ($> 35$ min), reflejando el incremento de armadura, resistencia mágica y agrupación defensiva de los equipos rivales.


---
## Lección 4: Combinación y estructuración de datasets

Integramos los tres conjuntos de datos limpios mediante su llave normalizada `champion_key`, auditando la relación entre tablas (`indicator=True`), validando la cardinalidad (`validate='m:1'` y `validate='1:1'`) y estructurando los datasets finales para las siguientes fases del proyecto.


### Paso 1: Auditoría de llaves e integridad referencial con `indicator=True`

Comprobamos que todas las observaciones de `meta_limpio` (96 registros) y `perf_limpio` (497 partidas válidas) encuentren correspondencia exacta en el catálogo primario `base_limpio` (70 campeones).


In [13]:
auditoria_meta_base = meta_limpio.merge(
    base_limpio[['champion_key']],
    on='champion_key',
    how='left',
    indicator=True
)
auditoria_perf_base = perf_limpio.merge(
    base_limpio[['champion_key']],
    on='champion_key',
    how='left',
    indicator=True
)

print("Auditoría de unión meta_limpio -> base_limpio:")
print(auditoria_meta_base['_merge'].value_counts().to_dict())

print("\nAuditoría de unión perf_limpio -> base_limpio:")
print(auditoria_perf_base['_merge'].value_counts().to_dict())


Auditoría de unión meta_limpio -> base_limpio:
{'both': 96, 'left_only': 0, 'right_only': 0}

Auditoría de unión perf_limpio -> base_limpio:
{'both': 497, 'left_only': 0, 'right_only': 0}


### Paso 2: Construcción de la Ficha Consolidada de Campeones y de la Tabla Maestra de Partidas (`merge` + `validate`)

Construimos dos estructuras analíticas complementarias:
1. **`ficha_campeones_lol` (`96 filas x 31 columnas`):** Combina cada registro de `meta_limpio` con los atributos de fábrica de `base_limpio` (`validate='m:1'`) y las métricas reales agregadas desde `perf_limpio` (`validate='m:1'`).
2. **`maestra_lol` (`497 filas x 40 columnas`):** Tabla maestra a nivel de partida que une cada encuentro clasificatorio de `perf_limpio` con los atributos base de diseño y el perfil competitivo del campeón (`validate='m:1'`).


In [14]:
# 1. Agregar desempeño real de las 497 partidas por campeón
resumen_partidas_campeon = (perf_limpio
    .groupby('champion_key')
    .agg(
        partidas_muestra=('match_id', 'count'),
        win_rate_muestra=('win', 'mean'),
        dano_medio_muestra=('damage_dealt', 'mean'),
        oro_medio_muestra=('gold_earned', 'mean'),
        duracion_media_min=('duration_min', 'mean'),
        dpm_medio_muestra=('dano_por_minuto', 'mean')
    )
    .round(3)
    .reset_index())

# 2. Construir Ficha Consolidada de Campeones (96 registros campeón-rol)
filas_meta_inicial = len(meta_limpio)
ficha_campeones_lol = (meta_limpio
    .merge(base_limpio.drop(columns=['name', 'primary_class']), on='champion_key', how='left', validate='m:1')
    .merge(resumen_partidas_campeon, on='champion_key', how='left', validate='m:1'))

assert len(ficha_campeones_lol) == filas_meta_inicial, 'Error de cardinalidad en ficha_campeones_lol'
assert not ficha_campeones_lol.isna().any().any(), 'Valores nulos detectados en ficha_campeones_lol'

# 3. Agregar resumen meta a nivel de campeón único (70 campeones) para unir m:1 con partidas
meta_resumen_campeon = (meta_limpio
    .sort_values('role_rate', ascending=False)
    .groupby('champion_key')
    .agg(
        clase_primaria=('champion_class', 'first'),
        rol_frecuente=('role', 'first'),
        roles_disponibles=('role', 'count'),
        tier_dominante=('tier', 'first'),
        meta_score_medio=('meta_score', 'mean'),
        win_rate_meta=('win_rate', 'mean'),
        pick_rate_meta=('pick_rate', 'sum'),
        ban_rate_meta=('ban_rate', 'mean'),
        kda_meta=('kda', 'mean')
    )
    .round(4)
    .reset_index())

# 4. Construir Tabla Analítica Maestra de Partidas (497 partidas clasificatorias válidas)
filas_perf_inicial = len(perf_limpio)
maestra_lol = (perf_limpio
    .merge(base_limpio.drop(columns=['name', 'primary_class']), on='champion_key', how='left', validate='m:1')
    .merge(meta_resumen_campeon, on='champion_key', how='left', validate='m:1'))

assert len(maestra_lol) == filas_perf_inicial, 'Error de cardinalidad en maestra_lol'
assert not maestra_lol.isna().any().any(), 'Valores nulos detectados en maestra_lol'

print(f"1. Ficha Consolidada de Campeones (ficha_campeones_lol): {ficha_campeones_lol.shape}")
print(f"2. Tabla Analítica Maestra de Partidas (maestra_lol):    {maestra_lol.shape}")
maestra_lol[['match_id', 'champion_name', 'champion_class', 'role', 'difficulty', 'tier_dominante', 'win', 'duration_min', 'dano_por_minuto', 'win_rate_meta']].head()


1. Ficha Consolidada de Campeones (ficha_campeones_lol): (96, 31)
2. Tabla Analítica Maestra de Partidas (maestra_lol):    (497, 40)
   match_id champion_name champion_class role  difficulty tier_dominante  win  duration_min  dano_por_minuto  win_rate_meta
0   4810001       Alistar           Tank  ADC           7              B    1     22.000000       816.400024         0.4918
1   4810001        Darius        Fighter  TOP           2              A    0     22.620001       999.400024         0.5058
2   4810002          Jhin       Marksman  TOP           6              A    1     23.230000      1173.099976         0.5198
3   4810002      Malphite           Tank  MID           2             S+    0     23.850000      1337.400024         0.5067
4   4810003         Riven        Fighter  MID           8              D    1     24.469999      1493.400024         0.4824


### Paso 3: Apilado con `pd.concat()` y transformación a formato largo con `.melt()`

1. **Uso de `pd.concat()`:** Unimos verticalmente los 3 campeones líderes de **Dificultad Baja (1–3)** y los 3 líderes de **Dificultad Alta (7–10)** en una sola tabla comparativa etiquetada.
2. **Uso de `.melt()`:** Transformamos las tasas competitivas promedio por clase (`win_rate_pct`, `pick_rate_pct`, `ban_rate_pct`) de formato ancho a formato largo (*tidy format*), facilitando su consumo directo en gráficos comparativos de Seaborn y Streamlit.


In [15]:
# 1. Apilado vertical con pd.concat()
lideres_baja_diff = (ficha_campeones_lol[ficha_campeones_lol['difficulty'] <= 3]
                     .nlargest(3, 'win_rate')[['champion_name', 'champion_class', 'role', 'difficulty', 'win_rate', 'ban_rate']]
                     .assign(perfil_complejidad='Líder Baja Dificultad (1-3)'))

lideres_alta_diff = (ficha_campeones_lol[ficha_campeones_lol['difficulty'] >= 7]
                     .nlargest(3, 'win_rate')[['champion_name', 'champion_class', 'role', 'difficulty', 'win_rate', 'ban_rate']]
                     .assign(perfil_complejidad='Líder Alta Dificultad (7-10)'))

comparativa_lideres_diff = pd.concat([lideres_baja_diff, lideres_alta_diff], ignore_index=True)
print("=== TABLA APILADA CON CONCAT: LÍDERES DE BAJA VS. ALTA DIFICULTAD ===")
print(comparativa_lideres_diff)

# 2. Reestructuración de formato ancho a largo con melt()
tasas_largo = resumen_por_clase.melt(
    id_vars=['champion_class'],
    value_vars=['win_rate_pct', 'pick_rate_pct', 'ban_rate_pct'],
    var_name='indicador_tasa',
    value_name='porcentaje'
)
print(f"\n=== FORMATO LARGO (MELT) PARA VISUALIZACIÓN: {tasas_largo.shape} ===")
tasas_largo.head(9)


=== TABLA APILADA CON CONCAT: LÍDERES DE BAJA VS. ALTA DIFICULTAD ===
  champion_name champion_class role  difficulty  win_rate  ban_rate            perfil_complejidad
0  Miss Fortune       Marksman  ADC           1    0.5418     0.141   Líder Baja Dificultad (1-3)
1         Fiora        Fighter  TOP           3    0.5373     0.061   Líder Baja Dificultad (1-3)
2        Darius        Fighter  ADC           2    0.5328     0.036   Líder Baja Dificultad (1-3)
3         Swain           Mage  MID           8    0.5409     0.060  Líder Alta Dificultad (7-10)
4          Yone       Assassin  MID           8    0.5400     0.359  Líder Alta Dificultad (7-10)
5          Azir           Mage  TOP           9    0.5391     0.278  Líder Alta Dificultad (7-10)

=== FORMATO LARGO (MELT) PARA VISUALIZACIÓN: (18, 3) ===
  champion_class indicador_tasa  porcentaje
0       Marksman   win_rate_pct   51.540001
1           Mage   win_rate_pct   50.650002
2        Fighter   win_rate_pct   50.560001
3       As

### Paso 4: Comparación de la calidad de los datos antes y después del procesamiento


In [16]:
comparacion_calidad = pd.DataFrame([
    {
        'Dataset': '1. Meta Stats',
        'Filas (Antes -> Después)': f"{len(df_meta_crudo)} -> {len(meta_limpio)}",
        'Nulos (Antes -> Después)': f"{int(df_meta_crudo.isna().sum().sum())} -> {int(meta_limpio.isna().sum().sum())}",
        'Memoria KB (Antes -> Después)': f"{df_meta_crudo.memory_usage(deep=True).sum()/1024:.1f} -> {meta_limpio.memory_usage(deep=True).sum()/1024:.1f}",
        'Transformaciones Clave': '4 pares (Name, Role) deduplicados; % a float32; Class desde tags; Tier moda (A)'
    },
    {
        'Dataset': '2. Base Attributes',
        'Filas (Antes -> Después)': f"{len(df_base_crudo)} -> {len(base_limpio)}",
        'Nulos (Antes -> Después)': f"{int(df_base_crudo.isna().sum().sum())} -> {int(base_limpio.isna().sum().sum())}",
        'Memoria KB (Antes -> Después)': f"{df_base_crudo.memory_usage(deep=True).sum()/1024:.1f} -> {base_limpio.memory_usage(deep=True).sum()/1024:.1f}",
        'Transformaciones Clave': 'Llave champion_key sin espacios/apóstrofos; mp/mpperlevel=0.0 en campeones sin maná'
    },
    {
        'Dataset': '3. Matches Performance',
        'Filas (Antes -> Después)': f"{len(df_perf_crudo)} -> {len(perf_limpio)}",
        'Nulos (Antes -> Después)': f"{int(df_perf_crudo.isna().sum().sum())} -> {int(perf_limpio.isna().sum().sum())}",
        'Memoria KB (Antes -> Después)': f"{df_perf_crudo.memory_usage(deep=True).sum()/1024:.1f} -> {perf_limpio.memory_usage(deep=True).sum()/1024:.1f}",
        'Transformaciones Clave': '3 partidas remake (<300s) filtradas; role y damage_dealt imputados; DPM y KDA calculados'
    },
    {
        'Dataset': '4. Tabla Maestra (maestra_lol)',
        'Filas (Antes -> Después)': f"N/A -> {len(maestra_lol)} (x{maestra_lol.shape[1]} cols)",
        'Nulos (Antes -> Después)': f"N/A -> {int(maestra_lol.isna().sum().sum())}",
        'Memoria KB (Antes -> Después)': f"N/A -> {maestra_lol.memory_usage(deep=True).sum()/1024:.1f}",
        'Transformaciones Clave': 'Integración relacional m:1 validada de partidas, atributos base y metajuego'
    }
])
comparacion_calidad


                          Dataset Filas (Antes -> Después) Nulos (Antes -> Después) Memoria KB (Antes -> Después)                             Transformaciones Clave
0                   1. Meta Stats                100 -> 96                  14 -> 0                   12.7 -> 6.2  4 pares (Name, Role) deduplicados; % a float32...
1              2. Base Attributes                 70 -> 70                  14 -> 0                   10.5 -> 7.5  Llave champion_key sin espacios/apóstrofos; mp...
2          3. Matches Performance               500 -> 497                  16 -> 0                  48.1 -> 33.9  3 partidas remake (<300s) filtradas; role y da...
3  4. Tabla Maestra (maestra_lol)    N/A -> 497 (x40 cols)                 N/A -> 0                   N/A -> 85.9  Integración relacional m:1 validada de partida...


### Paso 5: Exportación de resultados a `data/processed/` en formatos CSV y Parquet


In [17]:
# 1. Datasets limpios individuales
meta_limpio.to_csv(RUTA_PROCESSED / 'cleaned_lol_champion_meta.csv', index=False)
base_limpio.to_csv(RUTA_PROCESSED / 'cleaned_lol_base_attributes.csv', index=False)
perf_limpio.to_csv(RUTA_PROCESSED / 'cleaned_lol_matches_performance.csv', index=False)

# 2. Ficha consolidada de campeones y Tabla analítica maestra (CSV y Parquet)
ficha_campeones_lol.to_csv(RUTA_PROCESSED / 'lol_champions_consolidated.csv', index=False)
ficha_campeones_lol.to_parquet(RUTA_PROCESSED / 'lol_champions_consolidated.parquet', index=False)

maestra_lol.to_csv(RUTA_PROCESSED / 'lol_master_analytics.csv', index=False)
maestra_lol.to_parquet(RUTA_PROCESSED / 'lol_master_analytics.parquet', index=False)

# 3. Tablas resumen de indicadores e informe de auditoría
indicador_1_clase_rol.to_csv(RUTA_PROCESSED / 'resumen_clase_rol.csv', index=False)
indicador_2_dificultad.to_csv(RUTA_PROCESSED / 'resumen_dificultad.csv', index=False)
indicador_3_escalado.to_csv(RUTA_PROCESSED / 'resumen_escalado_duracion.csv', index=False)
informe_limpieza.to_csv(RUTA_PROCESSED / 'informe_limpieza_lol.csv', index=False)

print("Archivos procesados guardados y verificados en data/processed/:")
for p in sorted(RUTA_PROCESSED.glob('*lol*')) + sorted(RUTA_PROCESSED.glob('resumen_*')):
    print(f" - {p.name:<38} | {p.stat().st_size / 1024:>6.1f} KB")


Archivos procesados guardados y verificados en data/processed/:
 - cleaned_lol_base_attributes.csv        |    7.1 KB
 - cleaned_lol_champion_meta.csv          |    7.2 KB
 - cleaned_lol_matches_performance.csv    |   52.4 KB
 - informe_limpieza_lol.csv               |    0.4 KB
 - lol_champions_consolidated.csv         |   18.6 KB
 - lol_champions_consolidated.parquet     |   27.0 KB
 - lol_master_analytics.csv               |  114.4 KB
 - lol_master_analytics.parquet           |   52.7 KB
 - resumen_clase_rol.csv                  |    1.6 KB
 - resumen_departamentos.csv              |    0.8 KB
 - resumen_dificultad.csv                 |    0.4 KB
 - resumen_escalado_duracion.csv          |    1.4 KB


---
## 5. Conclusiones Finales frente a la Pregunta Analítica

1. **Influencia de la Clase y Rol (Indicador 1):**  
   El desempeño de cada clase depende estrechamente de su asignación de carril. Los **Tiradores (`Marksman`)** registran la mayor tasa de victoria promedio global (**51.54%**), alcanzando **52.14%** en su posición nativa (`ADC`) y **53.19%** en `MID`. Los **Luchadores (`Fighter`)** destacan como la clase más polivalente (> 50% de Win Rate en los 5 carriles), mientras que los **Asesinos (`Assassin`)** concentran la mayor presión de bloqueo del metajuego (**22.82% Ban Rate**), rindiendo al máximo en la `JUNGLE` (**51.26%**) pero cayendo a **47.88%** fuera de sus líneas naturales.
2. **Influencia de la Dificultad Mecánica (Indicador 2):**  
   Se comprueba empíricamente la **Paradoja de la Dificultad**: los campeones de **Baja Dificultad (1–3)** obtienen un **Win Rate promedio de 51.18%** frente al **50.57%** de **Media Dificultad (4–6)** y **50.33%** de **Alta Dificultad (7–10)**. La menor complejidad mecánica reduce los errores de ejecución en partidas clasificatorias, mientras que los campeones de alta dificultad concentran mayor selección (**10.92% Pick Rate**) y bloqueo (**20.64% Ban Rate**) por el impacto de sus especialistas (*Swain, Yone, Azir*).
3. **Influencia de la Duración del Encuentro (Indicador 3):**  
   Una vez filtradas las partidas canceladas (*remakes* $< 300$ s), las **497 partidas válidas** demuestran una clara especialización temporal: en partidas largas ($> 35$ min, con ~18,000 de oro), los **Tiradores (`Marksman`)** escalan hasta un **62.96% de Win Rate**, los **Magos (`Mage`)** alcanzan un **75.00% de Win Rate** y los **Tanques (`Tank`)** logran un **61.11% de Win Rate** (concentrando el **55.4%** de sus partidas en juego tardío). Por su parte, los **Asesinos (`Assassin`)** registran su mayor intensidad ofensiva en partidas cortas ($< 25$ min) con **1,200.8 DPM**, viendo reducido su ritmo de daño a **834.4 DPM** en partidas largas.
